# RAI Risk Taxonomy — 통합 파이프라인 (L4 중복 제거 → L3 배정)

**역할**: 🔵 노트북(결정론적 계산 전부) · 🟢 Claude(에이전트 판단 → 결정 JSON) · 🟠 교수(실행 + 승인)

Stage-1(L4 통합)과 Stage-2(L3 배정)를 한 파일에서 처리한다. 각 GATE는 결정 파일이 없으면 안내만 출력하고 멈춘다.

| 셀 | 담당 | 내용 |
|---|---|---|
| **P0–P2** | 🔵 | 설정 · 원본 로드 · 보호용어/수준 태깅 |
| **GATE-1** | 🟢→🟠 | 비리스크 판정 → 교수 승인 |
| P3 | 🔵 | 제거 적용 |
| **GATE-2** | 🟢 | 정규화 결정(형식만·기제 동결) |
| P4–P5 | 🔵 | 정규화 적용 + 위반 검사 · 임베딩 |
| **GATE-3** | 🟢 | 통합 사전(고유사 쌍 2인 판정) |
| P6 | 🔵 | 사전 선적용 · 제약 덴드로그램 · τ 절단 |
| **GATE-4** | 🟢 | 명명 검정(수준 위반 자동 기각) |
| P7–P8 | 🔵 | 세트 확정 · Stage-1 검증/산출물 |
| P9 | 🔵 | L3 로드 · 시드 보강 · 감쇠 EM 엔진 |
| P10 | 🔵 | EM 1단계(자동) + 민감도 3축×2회 |
| **GATE-5** | 🟢 | hold 심의 + 티어 상속 충돌 심의 |
| P11 | 🔵 | EM 2단계(강제) + 유효 민감도 |
| P12 | 🔵 | Stage-2 검증 · 산출물 · HTML |

**설계 원칙**
- 제거 기준은 "정의에 해악 실체 없음" 하나. 추상성·맥락 부재·능력 서술은 제거 사유 아님
- 정규화는 형식만. 해악 기제·결과 동결, 맥락 추가 금지, 보호 용어 개명 금지
- **교차 수준(construct↔instance) 병합 금지** — umbrella 카드 방지
- 병합 시 생존 ID·라벨·정의를 각각 독립 결정
- L3 배정은 **감쇠 EM**(α=0.4, 시드 앵커). 무작위 초기화는 비식별로 배제

In [ ]:
# P0 — 설정
import json, os, re, hashlib, platform, datetime, csv
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, scipy, pandas as pd
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

ROOT=Path.cwd()
while not (ROOT/'data/experiments/tau_tiers_v2_19/master_cards.json').exists():
    if ROOT.parent==ROOT: raise FileNotFoundError('repo root 못 찾음')
    ROOT=ROOT.parent
os.chdir(ROOT); print('repo root:',ROOT)

SRC=ROOT/'data/experiments/tau_tiers_v2_19/master_cards.json'
HIER=ROOT/'public/data/releases/v2.18.0-rc/hierarchy.json'
PREV=ROOT/'public/data/releases/v2.18.0-rc/cards.json'
TD=ROOT/'reports/consolidation/v2_19_tier_design'
EXP=ROOT/'data/experiments/pipeline'; DEC=EXP/'decisions'; OUT=EXP/'out'
for p in (EXP,DEC,OUT): p.mkdir(parents=True,exist_ok=True)
# 기존 결정 재사용 (있으면 그대로 사용)
for src,dst in ((ROOT/'data/experiments/stage1/decisions',DEC),(ROOT/'data/experiments/stage2/decisions',DEC)):
    if src.exists():
        for f in src.glob('*.json'):
            if not (dst/f.name).exists():
                (dst/f.name).write_bytes(f.read_bytes())
_m=ROOT/'tmp/models/bge-m3'
MODEL=str(_m) if (_m/'modules.json').exists() else 'BAAI/bge-m3'
if (_m/'modules.json').exists(): os.environ['HF_HUB_OFFLINE']='1'

CFG=dict(model=MODEL, max_seq_length=256, batch_size=32,
  linkage='complete', taus=dict(C80=0.80, C70=0.70), eps=0.001, block_cross_level=True,
  cannot_link=[('RAI4-0228','RAI4-0229'),('RAI4-0659','RAI4-0892'),('RAI4-0863','RAI4-0870'),
               ('RAI4-0682','RAI4-0691'),('RAI4-0435','RAI4-0569'),('RAI4-1275','RAI4-1307'),
               ('RAI4-1591','RAI4-1714')],
  alpha=0.4, alpha_alt=0.6, em_iters=30, em_tol=1e-5, margin_hold=0.03,
  boot_frac=0.9, boot_seeds=[20260807,20260808], seed_exemplars=8)
def sha(p): return hashlib.sha256(open(p,'rb').read()).hexdigest()[:16]
def unit(x): return x/np.maximum(np.linalg.norm(x,axis=-1,keepdims=True),1e-12)
print('model:',MODEL); print('DEC(🟢):',DEC); print('OUT(🔵):',OUT)

In [ ]:
# P1 — 원본 로드
doc=json.load(open(SRC))
cards=[dict(c) for c in doc['cards'] if c['status']=='active']
by={c['l4_id']:c for c in cards}
OLDMAP={c['l4_id']:c.get('primary_l3_id') for c in json.load(open(PREV))['cards']}
print('원본 활성 카드:',len(cards),'| sha',sha(SRC))

In [ ]:
# P2 — 보호 용어 + 수준(construct/instance) 태깅 [메타데이터만]
term=json.load(open(TD/'ko_term_dictionary.json'))
OFFICIAL={t['ko'].split('(')[0].strip() for t in term['terms']}
LITERATURE={'자동화 편향','목표 오정렬','통제 상실','권력 추구','환각','편향','견고성','설명 가능성',
 '가치 정렬','보상 해킹','명세 게이밍','기만','오용','인간 감독','적대적 공격','데이터 포이즈닝',
 '프롬프트 인젝션','탈옥','딥페이크','허위정보','개인정보 유출','목표의 잘못된 일반화','창발','책임 격차'}
GENERIC={'위험','안전','안전성','신뢰성','투명성','책임성','공정성','오용','기만','창발','견고성',
 '설명 가능성','편향','위험한 능력','인간 감독','가치 정렬','고영향 인공지능','생성형 인공지능',
 '개인정보 유출','허위정보','잘못된 정보'}
PROTECTED=sorted((OFFICIAL|LITERATURE)-GENERIC)
CTX=re.compile(r'(medical|clinical|health|driv|vehicle|hiring|recruit|credit|loan|court|judicial'
 r'|polic|welfare|educat|student|child|patient|worker|employee|military|weapon|financ|insur|border'
 r'|surveillance|robot|chatbot|search engine|recommend|moderation|consumer'
 r'|의료|임상|진료|운전|차량|채용|신용|대출|법원|사법|치안|복지|교육|학생|아동|환자|노동자|군사|무기'
 r'|금융|보험|국경|감시|로봇|자율주행|챗봇|추천|검열|소비자)', re.I)
def level(c):
    d=(c.get('definition_en') or '')+' '+(c.get('definition_ko') or '')
    return 'instance' if CTX.search(d) else 'construct'
for c in cards:
    c['level']=level(c); c['protected_term']=any(p in c['label_ko'] for p in PROTECTED)
BASE_LEVEL={c['l4_id']:c['level'] for c in cards}
print('수준:',dict(Counter(c['level'] for c in cards)),
      '| 보호용어 카드',sum(1 for c in cards if c['protected_term']),'| 보호용어',len(PROTECTED))
json.dump(PROTECTED,open(OUT/'protected_terms.json','w'),ensure_ascii=False,indent=1)
json.dump([{k:c.get(k) for k in ('l4_id','label_ko','label_en','definition_ko','definition_en','level','protected_term')}
           for c in cards], open(DEC/'_input_all_cards.json','w'), ensure_ascii=False, indent=0)

In [ ]:
# GATE-1 — 비리스크 판정 (🟢 Claude → 🟠 교수 승인)
P=DEC/'removals.json'
if not P.exists():
    print('⏸  대기: 🟢 Claude가 비리스크 판정 후 생성 ->',P)
    print('   기준: 정의에 해악 실체 없음 하나. 추상성·맥락 부재·능력 서술은 제거 사유 아님')
    print('   형식 {"approved": true, "removals":[{"l4_id":"...","reason":"..."}]}')
else:
    R=json.load(open(P)); rm={x['l4_id'] for x in R['removals']}
    assert R.get('approved') is True, '교수 승인(approved:true) 필요'
    assert rm <= set(by), '미존재 ID: '+str(sorted(rm-set(by))[:5])
    print('✅ 제거 승인:',len(rm),'장')

In [ ]:
# P3 — 제거 적용
rm={x['l4_id'] for x in json.load(open(DEC/'removals.json'))['removals']}
removed=[c for c in cards if c['l4_id'] in rm]
for c in removed: c['status']='retired'; c['retirement_reason']='no_harm_content'
work=[c for c in cards if c['l4_id'] not in rm]
by={c['l4_id']:c for c in work}
print(len(cards),'-',len(rm),'=',len(work))
json.dump([{k:c.get(k) for k in ('l4_id','label_ko','label_en','definition_ko','definition_en','level','protected_term')}
           for c in work], open(DEC/'_input_for_normalization.json','w'), ensure_ascii=False, indent=0)

In [ ]:
# GATE-2 — 정규화 결정 (🟢 Claude)
P=DEC/'normalization.json'
if not P.exists():
    print('⏸  대기: 🟢 Claude가 정규화 결정 후 생성 ->',P)
    print('   규칙: 기제·결과 동결 / 맥락 추가 금지 / 수준 변경 금지 / 보호용어 개명 금지')
    print('   형식 {"edits":[{"l4_id":"...","kind":"...","new_definition_ko":"...","new_definition_en":"...",')
    print('                  "new_label_ko":"(선택)","new_label_en":"(선택)"}]}')
else:
    E=json.load(open(P))['edits']
    print('✅ 정규화 결정',len(E),'건',dict(Counter(e.get('kind') for e in E)))

In [ ]:
# P4 — 정규화 적용 + 보호용어/수준 자동 검사
E=json.load(open(DEC/'normalization.json'))['edits']
viol=[]
for e in E:
    c=by.get(e['l4_id'])
    if c is None: continue
    nl=e.get('new_label_ko')
    if nl and c['protected_term'] and nl!=c['label_ko'] and not any(p in nl for p in PROTECTED):
        viol.append((c['l4_id'],c['label_ko'],nl))
assert not viol, '보호용어 개명 위반 '+str(len(viol))+'건: '+str(viol[:5])
for e in E:
    c=by.get(e['l4_id'])
    if c is None: continue
    for k in ('label_ko','label_en','definition_ko','definition_en'):
        if e.get('new_'+k):
            c.setdefault('original_'+k,c.get(k)); c[k]=e['new_'+k]
    c['normalization']=e.get('kind')
drift=[e['l4_id'] for e in E if e['l4_id'] in by and level(by[e['l4_id']])!=BASE_LEVEL[e['l4_id']]]
print('적용',len(E),'건 | 보호용어 위반 0 | 수준 변동',len(drift),'건',drift[:8])
json.dump(dict(release_id='pipeline-normalized',cards=work,removed_cards=removed),
          open(OUT/'normalized_master.json','w'),ensure_ascii=False,indent=1)

In [ ]:
# P5 — 임베딩 (텍스트 해시 캐시) + 고유사 쌍 추출
_enc=[None]
def enc():
    if _enc[0] is None:
        from sentence_transformers import SentenceTransformer
        m=SentenceTransformer(CFG['model']); m.max_seq_length=CFG['max_seq_length']; _enc[0]=m
    return _enc[0]
def embed(texts,tag):
    th=hashlib.sha1((tag+chr(30)+chr(30).join(texts)).encode()).hexdigest()[:12]
    p=OUT/('emb_'+tag+'_'+th+'.npy')
    if p.exists(): return np.load(p)
    v=enc().encode(texts,normalize_embeddings=True,batch_size=CFG['batch_size'],show_progress_bar=True).astype('float32')
    np.save(p,v); return v
def ctext(c,v='bilingual'):
    if v=='english': return (c.get('label_en','') or '')+'. '+(c.get('definition_en','') or '')
    return ((c.get('label_en','') or '')+'. '+(c.get('definition_en','') or '')+' / '
            +(c.get('label_ko','') or '')+'. '+(c.get('definition_ko','') or ''))
ids=[c['l4_id'] for c in work]
E_=embed([ctext(c) for c in work],'work_bilingual')
EMB_SHA=hashlib.sha1(E_.tobytes()).hexdigest()[:12]
En=E_/np.linalg.norm(E_,axis=1,keepdims=True); S=En@En.T; np.fill_diagonal(S,-1)
print('E',E_.shape,'sha1',EMB_SHA)
iu=np.triu_indices(len(ids),1); v=S[iu]; ordr=np.argsort(-v)
PROT={tuple(sorted(p)) for p in CFG['cannot_link']}
pairs=[]
for k in ordr[:600]:
    i,j=int(iu[0][k]),int(iu[1][k]); a,b=ids[i],ids[j]
    pairs.append(dict(a=a,b=b,cos=round(float(v[k]),4),
      level_a=by[a]['level'],level_b=by[b]['level'],cross_level=by[a]['level']!=by[b]['level'],
      protected=tuple(sorted((a,b))) in PROT,
      label_a=by[a]['label_ko'],label_b=by[b]['label_ko'],
      label_a_en=by[a]['label_en'],label_b_en=by[b]['label_en'],
      def_a=(by[a].get('definition_ko') or '')[:220],def_b=(by[b].get('definition_ko') or '')[:220]))
json.dump(pairs,open(DEC/'_input_top_pairs.json','w'),ensure_ascii=False,indent=1)
print('-> 🟢 상위 600쌍 | cos>=0.85',int((v>=0.85).sum()),'| >=0.80',int((v>=0.80).sum()))

In [ ]:
# GATE-3 — 통합 사전 (🟢 Claude)
P=DEC/'merge_dictionary.json'
if not P.exists():
    print('⏸  대기: 🟢 Claude가 고유사 쌍 2인 판정 후 생성 ->',P)
    print('   형식 {"merges":[{"ids":[...],"survivor_id":"...","label_ko":"...","label_en":"...",')
    print('                   "definition_ko":"...","definition_en":"..."}]}')
else:
    D=json.load(open(P))['merges']
    bad=[m['ids'] for m in D for p in PROT if p[0] in m['ids'] and p[1] in m['ids']]
    assert not bad, '보호 쌍 흡수: '+str(bad[:3])
    xl=[m['ids'] for m in D if len({by[i]['level'] for i in m['ids'] if i in by})>1]
    assert not (xl and CFG['block_cross_level']), '교차 수준 병합: '+str(xl[:3])
    used=[i for m in D for i in m['ids']]
    assert len(used)==len(set(used)), '그룹 간 카드 중복'
    print('✅ 사전',len(D),'그룹 /',len(used),'장 -> 순감',len(used)-len(D))

In [ ]:
# P6 — 사전 선적용 + 제약 덴드로그램 + τ 절단 + 명명 후보
D=json.load(open(DEC/'merge_dictionary.json'))['merges']
LEDGER={}; _c=[1900]
def issue(mem):
    k=tuple(sorted(mem))
    for a,b in LEDGER.items():
        if b==k: return a
    n='RAI4-'+format(_c[0],'04d'); _c[0]+=1; LEDGER[n]=k; return n
def amax(srcs,key):
    vs=[s.get(key) for s in srcs if s.get(key) is not None]
    return round(max(vs),3) if vs else None
merged=[]
for m in sorted(D,key=lambda x:sorted(x['ids'])[0]):
    srcs=[by[i] for i in m['ids']]
    refs,seen=[],set()
    for s in srcs:
        for r in (s.get('references') or []):
            k=(r.get('title'),r.get('url'))
            if k not in seen: seen.add(k); refs.append(dict(r,source_l4_id=s['l4_id']))
    nid=issue(m['ids'])
    merged.append(dict(l4_id=nid,label_ko=m['label_ko'],label_en=m['label_en'],
      definition_ko=m['definition_ko'],definition_en=m['definition_en'],
      severity_1to5=amax(srcs,'severity_1to5'),probability_0to1=amax(srcs,'probability_0to1'),
      metrics_note='max over members; probability lower bound',
      member_metrics=[{'l4_id':s['l4_id'],'s':s.get('severity_1to5'),'p':s.get('probability_0to1')} for s in srcs],
      references=refs,status='active',stage1_source_ids=sorted(m['ids']),
      level=Counter(s['level'] for s in srcs).most_common(1)[0][0],
      protected_term=any(s['protected_term'] for s in srcs),merge_basis='dictionary'))
    for s in srcs: s['status']='retired'; s['merged_into']=nid
W=[c for c in work if c['status']=='active']+merged
byW={c['l4_id']:c for c in W}; wids=[c['l4_id'] for c in W]
print('사전 적용:',len(D),'그룹 ->',len(W),'장')

emap={ids[i]:En[i] for i in range(len(ids))}
new=[c for c in W if c['l4_id'] not in emap]
if new:
    V=embed([ctext(c) for c in new],'merged_'+str(len(new)))
    for c,vv in zip(new,V): emap[c['l4_id']]=vv
EW=np.vstack([emap[i] for i in wids]).astype('float32')
EWn=EW/np.linalg.norm(EW,axis=1,keepdims=True); SW=EWn@EWn.T; np.fill_diagonal(SW,1.0)
Dm=np.clip(1-SW,0,None); np.fill_diagonal(Dm,0)
idx={k:i for i,k in enumerate(wids)}
def resolve(x):
    while x not in idx:
        nx=[k for k,vv in LEDGER.items() if x in vv]
        if not nx: raise ValueError('cannot-link '+x+' 소실')
        x=nx[0]
    return x
CL=[]
for a,b in CFG['cannot_link']:
    ra,rb=resolve(a),resolve(b)
    if ra==rb: raise ValueError('보호 쌍 흡수: '+a+','+b)
    CL.append((ra,rb)); Dm[idx[ra],idx[rb]]=Dm[idx[rb],idx[ra]]=10.0
if CFG['block_cross_level']:
    lv=np.array([1 if byW[i]['level']=='construct' else 0 for i in wids])
    mask=lv[:,None]!=lv[None,:]; Dm[mask]=10.0; np.fill_diagonal(Dm,0)
    print('교차 수준 차단 쌍:',int(mask.sum())//2)
Z=linkage(squareform(Dm,checks=False),method=CFG['linkage'])
S2=SW.copy(); np.fill_diagonal(S2,-1)
for a,b in CL: S2[idx[a],idx[b]]=S2[idx[b],idx[a]]=-1
tau_nm=float(S2.max())+CFG['eps']
cuts={}
for tier,tau in CFG['taus'].items():
    lab=fcluster(Z,t=1-tau,criterion='distance')
    g=defaultdict(list)
    for i,x in enumerate(lab): g[int(x)].append(wids[i])
    cuts[tier]=dict(labels=lab,groups=dict(g))
    print(tier+':',len(g),'cards, multi',sum(1 for vv in g.values() if len(vv)>1),
          ', largest',max(len(vv) for vv in g.values()))
print('tau_nm =',format(tau_nm,'.4f'))
json.dump([(round(float(t),2),len(set(fcluster(Z,t=1-t,criterion='distance')))) for t in np.arange(0.60,0.96,0.01)],
          open(OUT/'tau_sweep.json','w'))
FP=hashlib.sha1((hashlib.sha1(EW.tobytes()).hexdigest()+json.dumps(sorted(wids))).encode()).hexdigest()[:16]
for tier in cuts:
    rows=[dict(group=g,members_key=sorted(vv),level=byW[vv[0]]['level'],
               members=[dict(l4_id=i,label_ko=byW[i]['label_ko'],label_en=byW[i]['label_en'],
                             definition_ko=(byW[i].get('definition_ko') or '')[:300],
                             protected=byW[i]['protected_term']) for i in vv])
          for g,vv in sorted(cuts[tier]['groups'].items()) if len(vv)>1]
    json.dump(dict(fingerprint=FP,tier=tier,groups=rows),
              open(DEC/('_input_naming_'+tier+'.json'),'w'),ensure_ascii=False,indent=1)
    print('-> 🟢',tier,len(rows),'다중 군집')

In [ ]:
# GATE-4 — 명명 검정 (🟢 Claude)
missing=[t for t in ('C80','C70') if not (DEC/('naming_'+t+'.json')).exists()]
if missing:
    print('⏸  대기: 🟢 Claude가 명명 검정 후 생성 ->',['naming_'+t+'.json' for t in missing])
    print('   fingerprint =',FP)
    print('   규칙: 상위어 명사구 영문 12단어 이내 / 나열식·공허 포괄어 기각 /')
    print('        제안 명칭이 확립 구성개념 명칭 수준이면 기각(수준 위반)')
else:
    for t in ('C80','C70'):
        d=json.load(open(DEC/('naming_'+t+'.json')))
        assert d['fingerprint']==FP, t+' fingerprint 불일치 — 재검수 필요'
        a=sum(1 for x in d['decisions'] if x['verdict']=='approve')
        print('✅',t+':',len(d['decisions']),'결정 (승인',a,', 기각',len(d['decisions'])-a,')')

In [ ]:
# P7 — 명명 결정 적용 → C80/C70 확정 (C70 기각을 C80에 전파해 ⊇ 유지)
def loadn(t):
    d=json.load(open(DEC/('naming_'+t+'.json'))); assert d['fingerprint']==FP
    o={}
    for x in d['decisions']:
        g=int(x['group']); assert sorted(x['members_key'])==sorted(cuts[t]['groups'][g])
        if x['verdict']=='approve':
            assert all(x.get(k) for k in ('label_ko','label_en','definition_ko','definition_en'))
        o[g]=x
    return o
d70,d80=loadn('C70'),loadn('C80')
rej70={i for g,mem in cuts['C70']['groups'].items()
       if len(mem)>1 and d70.get(g,{}).get('verdict')!='approve' for i in mem}
def srcof(i): return sorted(set(byW[i].get('stage1_source_ids') or [i]))
tiers={}
for tier,dec in (('C80',d80),('C70',d70)):
    tc=[]; miss=0
    for g,mem in sorted(cuts[tier]['groups'].items()):
        if len(mem)==1:
            tc.append(dict(byW[mem[0]],master_source_ids=srcof(mem[0]))); continue
        x=dec.get(g)
        if x is None: miss+=1
        if (x is None) or x['verdict']!='approve' or (tier=='C80' and set(mem)&rej70):
            tc+=[dict(byW[i],master_source_ids=srcof(i)) for i in mem]; continue
        srcs=[byW[i] for i in mem]
        refs,seen=[],set()
        for s in srcs:
            for r in (s.get('references') or []):
                k=(r.get('title'),r.get('url'))
                if k not in seen: seen.add(k); refs.append(dict(r,source_l4_id=r.get('source_l4_id',s['l4_id'])))
        tc.append(dict(l4_id=issue(mem),label_ko=x['label_ko'],label_en=x['label_en'],
            definition_ko=x['definition_ko'],definition_en=x['definition_en'],
            severity_1to5=amax(srcs,'severity_1to5'),probability_0to1=amax(srcs,'probability_0to1'),
            metrics_note='max over members; probability lower bound',
            member_metrics=[{'l4_id':s['l4_id'],'s':s.get('severity_1to5'),'p':s.get('probability_0to1')} for s in srcs],
            references=refs,master_source_ids=sorted({y for i in mem for y in srcof(i)}),
            level=srcs[0]['level'],merge_basis='nameability_'+tier,status='active'))
    tiers[tier]=tc; print(tier+':',len(tc),'cards (결정 누락',miss,')')

In [ ]:
# P8 — Stage-1 검증 + 산출물
ref={c['l4_id'] for c in work}
ok1=True
for tier,tc in tiers.items():
    lab=cuts[tier]['labels']
    assert not [(a,b) for a,b in CL if lab[idx[a]]==lab[idx[b]]], tier+' cannot-link 위반'
    mp={s for c in tc for s in c['master_source_ids']}
    assert mp==ref, tier+' 매핑 불일치'
    xl=[c['l4_id'] for c in tc if len(c['master_source_ids'])>1
        and len({by[s]['level'] for s in c['master_source_ids'] if s in by})>1]
    dupk=[k for k,v in Counter(c['label_ko'] for c in tc).items() if v>1]
    dupe=[k for k,v in Counter(c['label_en'].lower() for c in tc).items() if v>1]
    lng=[c['l4_id'] for c in tc if len(c['label_ko'])>28 or len(c['label_en'].split())>12]
    print(f'{tier}: {len(tc)}장 | cannot-link 0 | 매핑무손실 | 교차수준 {len(xl)} | 라벨중복 한{len(dupk)}/영{len(dupe)} | 길이위반 {len(lng)}')
    if xl or dupk or dupe or lng: ok1=False
m70={s:c['l4_id'] for c in tiers['C70'] for s in c['master_source_ids']}
nest=[c['l4_id'] for c in tiers['C80'] if len({m70[s] for s in c['master_source_ids']})>1]
print('중첩 위반:',len(nest)); ok1 = ok1 and not nest
json.dump(dict(release_id='pipeline-master',cards=work,removed_cards=removed,
               id_ledger={k:list(v) for k,v in LEDGER.items()}),
          open(OUT/'master.json','w'),ensure_ascii=False,indent=1)
for tier,tc in tiers.items():
    json.dump(dict(release_id='pipeline-'+tier,tau=CFG['taus'][tier],cards=tc),
              open(OUT/(tier.lower()+'.json'),'w'),ensure_ascii=False,indent=1)
    with open(OUT/(tier.lower()+'_mapping.csv'),'w',newline='') as f:
        w=csv.writer(f); w.writerow(['master_l4_id','tier_l4_id'])
        for c in tc:
            for s in c['master_source_ids']: w.writerow([s,c['l4_id']])
print('\nStage-1:','PASS' if ok1 else 'FAIL','| Master',len(work),'C80',len(tiers['C80']),'C70',len(tiers['C70']))

In [ ]:
# P9 — L3 로드 + 시드 보강 + 감쇠 EM 엔진
H=json.load(open(HIER))
L3=[n for n in H['nodes'] if str(n['node_id']).startswith('RAI3')]
HOLD_L3={n['node_id'] for n in L3 if 'HLD' in n['node_id']}
l3idx={n['node_id']:i for i,n in enumerate(L3)}
HOLD_MASK=np.array([n['node_id'] in HOLD_L3 for n in L3]); NEG=np.where(HOLD_MASK,-1e9,0.0)
print('L3',len(L3),'| HLD(배정 후보 제외)',sorted(HOLD_L3))

SETS={'master':work,'c80':tiers['C80'],'c70':tiers['C70']}
CARD={}
for s,cs in SETS.items():
    for v in ('bilingual','english'):
        CARD[(s,v)]=embed([ctext(c,v) for c in cs],'l3_'+s+'_'+v)
def l3_plain(n):
    return ((n.get('label_en','') or '')+'. '+(n.get('definition_en','') or '')+' / '
            +(n.get('label_ko','') or '')+'. '+(n.get('definition_ko','') or ''))
SEED_PLAIN=embed([l3_plain(n) for n in L3],'l3seed_plain')
Xm=CARD[('master','bilingual')]
prevmap=defaultdict(list)
for i,c in enumerate(work):
    p=OLDMAP.get(c['l4_id'])
    if p and p not in HOLD_L3: prevmap[p].append(i)
ex=[]
for j,n in enumerate(L3):
    b=l3_plain(n); cand=prevmap.get(n['node_id'],[])
    if n['node_id'] in HOLD_L3 or not cand: ex.append(b); continue
    top=[cand[t] for t in np.argsort(-(Xm[cand]@SEED_PLAIN[j]))[:CFG['seed_exemplars']]]
    ex.append(b+' Examples: '+' ; '.join((work[t].get('label_en') or '')+' / '+(work[t].get('label_ko') or '') for t in top))
SEED_ENR=embed(ex,'l3seed_enriched')
SEEDS={'plain':SEED_PLAIN,'enriched':SEED_ENR}
print('시드 보강 완료 | plain-enriched 코사인 평균',round(float((SEED_PLAIN*SEED_ENR).sum(1).mean()),3))

def dem(X,SD,alpha,iters=None,tol=None,fixed=None,fixedA=None):
    iters=iters or CFG['em_iters']; tol=tol or CFG['em_tol']
    MU=SD.copy(); A=None
    for it in range(1,iters+1):
        Sm=X@MU.T+NEG; nA=Sm.argmax(1)
        if fixed is not None: nA[fixed]=fixedA[fixed]
        NEW=MU.copy()
        for k in range(len(SD)):
            m=X[nA==k]
            NEW[k]=unit(alpha*SD[k]+(1-alpha)*unit(m.mean(0))) if len(m) else SD[k]
        stop=np.allclose(NEW,MU,atol=tol) and A is not None and (nA==A).all()
        MU=NEW; A=nA
        if stop: break
    Sm=X@MU.T+NEG; A=Sm.argmax(1)
    if fixed is not None: A[fixed]=fixedA[fixed]
    srt=np.sort(Sm,1)
    return dict(assign=A,margin=srt[:,-1]-srt[:,-2],sim=Sm,top3=np.argsort(-Sm,1)[:,:3],
                iters=it,drift=(MU*SD).sum(1))
A_seed=(Xm@SEED_ENR.T+NEG).argmax(1)
print('\nα 진단 (master)')
for a in (0.0,0.2,0.4,0.6,0.8):
    r=dem(Xm,SEED_ENR,a)
    print(f'  α={a}: iters {r["iters"]:>2} | 시드일치 {(r["assign"]==A_seed).mean()*100:5.1f}%'
          f' | drift_min {r["drift"].min():.3f} | med margin {np.median(r["margin"]):.4f}'
          f' | max군집 {Counter(r["assign"].tolist()).most_common(1)[0][1]}')

In [ ]:
# P10 — EM 1단계(자동) + 민감도 3축 × 2회(90% 부트스트랩)
AXES=[('base','bilingual','enriched',CFG['alpha']),
      ('axis1_english','english','enriched',CFG['alpha']),
      ('axis2_plainseed','bilingual','plain',CFG['alpha']),
      ('axis3_alpha06','bilingual','enriched',CFG['alpha_alt'])]
rows=[]; RES1={}
for s in SETS:
    for ax,cv,sv,al in AXES:
        Xf=CARD[(s,cv)]; SD=SEEDS[sv]
        full=dem(Xf,SD,al); RES1[(s,ax)]=full; mg=full['margin']
        rows.append(dict(set=s,axis=ax,run='full',n=len(Xf),em_iters=full['iters'],
            median_margin=round(float(np.median(mg)),4),
            hold=int((mg<CFG['margin_hold']).sum()),
            hold_pct=round(float((mg<CFG['margin_hold']).mean()*100),1),
            used_l3=len(set(full['assign'])),max_family=Counter(full['assign'].tolist()).most_common(1)[0][1],
            drift_min=round(float(full['drift'].min()),3)))
        for bs in CFG['boot_seeds']:
            rng=np.random.default_rng(bs)
            sel=np.sort(rng.choice(len(Xf),int(len(Xf)*CFG['boot_frac']),replace=False))
            rb=dem(Xf[sel],SD,al)
            rows.append(dict(set=s,axis=ax,run='boot'+str(bs),n=len(sel),em_iters=rb['iters'],
                median_margin=round(float(np.median(rb['margin'])),4),
                hold=int((rb['margin']<CFG['margin_hold']).sum()),
                hold_pct=round(float((rb['margin']<CFG['margin_hold']).mean()*100),1),
                used_l3=len(set(rb['assign'])),max_family=Counter(rb['assign'].tolist()).most_common(1)[0][1],
                drift_min=round(float(rb['drift'].min()),3),
                boot_agreement_pct=round(float((rb['assign']==full['assign'][sel]).mean()*100),1)))
df1=pd.DataFrame(rows); df1.to_csv(OUT/'em_stage1_sensitivity.csv',index=False)
print(df1[df1.run=='full'].to_string(index=False))
try: from sklearn.metrics import adjusted_rand_score as ari
except ImportError:
    def ari(a,b):
        n=len(a); c=Counter(zip(a,b)); sa=Counter(a); sb=Counter(b); cmb=lambda x:x*(x-1)/2
        s=sum(cmb(v) for v in c.values()); A=sum(cmb(v) for v in sa.values()); B=sum(cmb(v) for v in sb.values())
        e=A*B/cmb(n); m=(A+B)/2
        return (s-e)/(m-e) if m!=e else 1.0
st=[dict(set=s,vs=ax,agreement_pct=round(float((RES1[(s,ax)]['assign']==RES1[(s,'base')]['assign']).mean()*100),1),
         ARI=round(float(ari(RES1[(s,'base')]['assign'].tolist(),RES1[(s,ax)]['assign'].tolist())),3))
    for s in SETS for ax,_,_,_ in AXES if ax!='base']
pd.DataFrame(st).to_csv(OUT/'em_stage1_stability.csv',index=False)
print(); print(pd.DataFrame(st).to_string(index=False))

STAGE1={}
for s in SETS:
    r=RES1[(s,'base')]; A=r['assign']; mg=r['margin']
    STAGE1[s]=[dict(l4_id=SETS[s][i]['l4_id'],
        l3_stage1=None if mg[i]<CFG['margin_hold'] else L3[A[i]]['node_id'],
        margin=round(float(mg[i]),4),hold=bool(mg[i]<CFG['margin_hold']),
        prev_l3=OLDMAP.get(SETS[s][i]['l4_id']),
        top3=[dict(l3=L3[j]['node_id'],label_ko=L3[j]['label_ko'],sim=round(float(r['sim'][i,j]),4))
              for j in r['top3'][i]]) for i in range(len(A))]
    h=sum(1 for x in STAGE1[s] if x['hold'])
    print(f'{s}: 자동 {len(A)-h} / hold {h} ({h/len(A)*100:.1f}%)')
json.dump(STAGE1,open(OUT/'em_stage1_assignment.json','w'),ensure_ascii=False,indent=1)
for s in SETS:
    bc={c['l4_id']:c for c in SETS[s]}
    json.dump([dict(l4_id=x['l4_id'],label_ko=bc[x['l4_id']]['label_ko'],label_en=bc[x['l4_id']]['label_en'],
        definition_ko=bc[x['l4_id']].get('definition_ko'),definition_en=bc[x['l4_id']].get('definition_en'),
        margin=x['margin'],top3=x['top3'],prev_l3=x['prev_l3']) for x in STAGE1[s] if x['hold']],
        open(DEC/('_input_holds_'+s+'.json'),'w'),ensure_ascii=False,indent=1)
json.dump([dict(node_id=n['node_id'],label_ko=n['label_ko'],label_en=n['label_en'],
                definition_ko=n.get('definition_ko'),definition_en=n.get('definition_en'),
                parent_id=n['parent_id'],is_hold=n['node_id'] in HOLD_L3) for n in L3],
          open(DEC/'_input_l3_catalog.json','w'),ensure_ascii=False,indent=1)
print('-> 🟢 hold 입력 + L3 카탈로그 생성')

In [ ]:
# GATE-5 — hold 심의 + 티어 상속 충돌 심의 (🟢 Claude)
P=DEC/'hold_decisions.json'
if not P.exists():
    print('⏸  대기: 🟢 Claude가 hold 심의 후 생성 ->',P)
    print('   원칙: master는 hold를 전수 심의(HLD 노드 배정 금지),')
    print('        티어는 master 상속 + 동률 충돌만 별도 심의')
    print('   형식 {"decisions":{"master":[{"l4_id":"...","l3":"RAI3-..."}],"c80":[...],"c70":[...]}}')
else:
    HD=json.load(open(P))['decisions']; valid={n['node_id'] for n in L3}
    for s in SETS:
        assert not [d for d in HD.get(s,[]) if d['l3'] not in valid or d['l3'] in HOLD_L3], s+' 잘못된 L3'
        ids={x['l4_id'] for x in STAGE1[s] if x['hold']}; got={d['l4_id'] for d in HD.get(s,[])}
        print(f'✅ {s}: hold {len(ids)} / 결정 {len(got)} / 미심의 {len(ids-got)}')

In [ ]:
# P11 — EM 2단계(강제 배정) + 유효 민감도 (심의분만 고정)
HD=json.load(open(DEC/'hold_decisions.json'))['decisions']
FINAL={}; SENS2=[]
for s in SETS:
    man={d['l4_id']:d['l3'] for d in HD.get(s,[])}
    Xf=CARD[(s,'bilingual')]; SD=SEEDS['enriched']
    fixed=np.array([c['l4_id'] in man for c in SETS[s]])
    fixedA=np.array([l3idx[man[c['l4_id']]] if c['l4_id'] in man else 0 for c in SETS[s]])
    r=dem(Xf,SD,CFG['alpha'],fixed=fixed,fixedA=fixedA)
    A=r['assign']; mg=r['margin']
    src=['adjudicated' if fixed[i] else ('auto' if not STAGE1[s][i]['hold'] else 'forced') for i in range(len(A))]
    FINAL[s]=[dict(l4_id=SETS[s][i]['l4_id'],l3=L3[A[i]]['node_id'],l3_label_ko=L3[A[i]]['label_ko'],
                   margin=round(float(mg[i]),4),source=src[i],prev_l3=OLDMAP.get(SETS[s][i]['l4_id']))
              for i in range(len(A))]
    print(f'{s}: {len(A)}장 | {dict(Counter(src))} | usedL3 {len(set(A))}/{len(L3)}'
          f' | max군집 {Counter(A.tolist()).most_common(1)[0][1]}')
    # 유효 민감도: 심의(고정) 제외한 자유 카드 기준
    free=~fixed
    if free.sum()==0:
        print('   ※ 전 카드 고정 — 민감도 검정 불가(자유 카드 0). 결정 파일을 hold만 담도록 축소 권장')
        continue
    base=A
    for ax,cv,sv,al in AXES:
        if ax=='base': continue
        a2=dem(CARD[(s,cv)],SEEDS[sv],al,fixed=fixed,fixedA=fixedA)['assign']
        SENS2.append(dict(set=s,condition=ax,all_pct=round(float((a2==base).mean()*100),1),
                          free_pct=round(float((a2[free]==base[free]).mean()*100),1)))
    for bs in CFG['boot_seeds']:
        rng=np.random.default_rng(bs)
        sel=np.sort(rng.choice(len(Xf),int(len(Xf)*CFG['boot_frac']),replace=False))
        a2=dem(Xf[sel],SD,CFG['alpha'],fixed=fixed[sel],fixedA=fixedA[sel])['assign']
        fs=free[sel]
        SENS2.append(dict(set=s,condition='boot'+str(bs),
            all_pct=round(float((a2==base[sel]).mean()*100),1),
            free_pct=round(float((a2[fs]==base[sel][fs]).mean()*100),1) if fs.sum() else None))
json.dump(FINAL,open(OUT/'em_stage2_assignment.json','w'),ensure_ascii=False,indent=1)
if SENS2:
    df2=pd.DataFrame(SENS2); df2.to_csv(OUT/'em_stage2_sensitivity.csv',index=False)
    print(); print(df2.to_string(index=False))

In [ ]:
# P12 — Stage-2 검증 + 산출물 + HTML
import html as HH
valid={n['node_id'] for n in L3}; ok2=True
for s in SETS:
    F=FINAL[s]
    assert len({x['l4_id'] for x in F})==len(SETS[s])
    assert all(x['l3'] in valid for x in F)
    hl=[x for x in F if x['l3'] in HOLD_L3]
    empty=[n['node_id'] for n in L3 if n['node_id'] not in HOLD_L3
           and n['node_id'] not in {x['l3'] for x in F}]
    print(f'{s}: {len(F)}장 전원 배정 | HLD 잔류 {len(hl)} | 빈 실질 L3 {len(empty)}')
    if hl: ok2=False
mas={x['l4_id']:x['l3'] for x in FINAL['master']}
for t in ('c80','c70'):
    tm={x['l4_id']:x['l3'] for x in FINAL[t]}
    mism=sum(1 for c in SETS[t]
             if len({mas[i] for i in c['master_source_ids'] if i in mas})==1
             and tm[c['l4_id']] not in {mas[i] for i in c['master_source_ids'] if i in mas})
    print(f'{t}: 단일출처 L3 불일치 {mism}'); ok2 = ok2 and not mism
for s in SETS:
    with open(OUT/(s+'_l3_mapping.csv'),'w',newline='') as f:
        w=csv.writer(f); w.writerow(['l4_id','l3_id','l3_label_ko','margin','source','prev_l3'])
        for x in FINAL[s]: w.writerow([x['l4_id'],x['l3'],x['l3_label_ko'],x['margin'],x['source'],x['prev_l3']])
man=dict(run_at=datetime.datetime.now().isoformat(),cfg=CFG,
    method='Stage-1 level-constrained complete linkage + nameability gate; Stage-2 damped seed-anchored EM (random init excluded as non-identified)',
    inputs={str(SRC):sha(SRC),str(HIER):sha(HIER)},embedding_sha1=EMB_SHA,fingerprint=FP,tau_nm=round(tau_nm,4),
    versions=dict(python=platform.python_version(),numpy=np.__version__,scipy=scipy.__version__),
    counts=dict(source=len(cards),master=len(work),**{t:len(c) for t,c in tiers.items()}),
    l3_used={s:len({x['l3'] for x in FINAL[s]}) for s in SETS})
json.dump(man,open(OUT/'manifest.json','w'),ensure_ascii=False,indent=1,default=str)
cnt={s:Counter(x['l3'] for x in FINAL[s]) for s in SETS}
tr=''.join('<tr><td><code>'+n['node_id']+'</code></td><td>'+HH.escape(n['label_ko'])+'</td>'
           +''.join('<td class=num>'+str(cnt[t].get(n['node_id'],0))+'</td>' for t in SETS)+'</tr>' for n in L3)
CSS=("<style>body{font-family:'Apple SD Gothic Neo','Noto Sans KR',sans-serif;margin:18px}"
     "table{border-collapse:collapse;width:100%;font-size:12.5px}th,td{border:1px solid #ddd;padding:6px 8px}"
     "th{background:#f4f4f2}.num{text-align:right}code{color:#888}</style>")
open(OUT/'result.html','w').write('<!doctype html><meta charset=utf-8><title>Pipeline 결과</title>'+CSS
    +'<h1>통합 파이프라인 결과</h1><p>'
    +' · '.join(s.upper()+' '+str(len(FINAL[s]))+'장' for s in SETS)
    +' · 감쇠 EM α='+str(CFG['alpha'])+' · 보강 시드 · tau_nm '+format(tau_nm,'.4f')+'</p>'
    +'<table><tr><th>L3</th><th>명칭</th>'+''.join('<th>'+s.upper()+'</th>' for s in SETS)+'</tr>'+tr+'</table>')
print('\nStage-1',('PASS' if ok1 else 'FAIL'),'| Stage-2',('PASS' if ok2 else 'FAIL'),'->',OUT)

## 셀 20 — L4 2차원 지형도 (C80/C70, L3 색상)
UMAP(cosine, n_neighbors=25, min_dist=0.35, seed=42)으로 BGE-M3 이중언어 임베딩을 축소해 `data/experiments/review/{c80,c70}_landscape.png` 저장.

In [ ]:
# L4 카드 2차원 임베딩 지형도 (C80 / C70) — L3 색상, UMAP
# 노트북 셀로도 동일 코드 사용 (셀 20)
import json, glob, numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import umap
from collections import defaultdict

S1='data/experiments/stage1/out'; S2='data/experiments/stage2/out'; OUT='data/experiments/review'
HI=json.load(open('public/data/releases/v2.18.0-rc/hierarchy.json'))
nd={n['node_id']:n for n in HI['nodes']}
FINAL=json.load(open(f'{S2}/stage2_assignment.json'))

# L2별 색상 계열 (첨부 예시처럼 상위 그룹별 색조 통일)
L2_BASE={'RAI2-G-INT':plt.cm.Purples,'RAI2-G-SYS':plt.cm.Blues,'RAI2-G-SOC':plt.cm.RdPu,
         'RAI2-A-SYS':plt.cm.Greens,'RAI2-P-INT':plt.cm.Oranges,'RAI2-P-SYS':plt.cm.YlOrBr}

def l3_colors(l3_ids):
    by_l2=defaultdict(list)
    for l in sorted(l3_ids): by_l2[nd[l]['parent_id']].append(l)
    cmap={}
    for l2,ls in by_l2.items():
        cm=L2_BASE.get(l2, plt.cm.Greys)
        for i,l in enumerate(ls): cmap[l]=cm(0.35+0.6*i/max(1,len(ls)-1))
    return cmap

def build(setname, seed=42):
    cards=json.load(open(f'{S1}/{setname}.json'))['cards']
    emb=np.load(glob.glob(f'{S2}/emb_{setname}_bilingual_*.npy')[0])
    assert len(cards)==emb.shape[0], '카드-임베딩 행 수 불일치'
    l3of={x['l4_id']:x['l3'] for x in FINAL[setname]}
    labs=[l3of[c['l4_id']] for c in cards]
    xy=umap.UMAP(n_neighbors=25, min_dist=0.35, metric='cosine',
                 random_state=seed).fit_transform(emb)
    cmap=l3_colors(set(labs))
    fig,ax=plt.subplots(figsize=(13,11),dpi=150)
    for l in sorted(set(labs)):
        m=np.array([x==l for x in labs])
        ax.scatter(xy[m,0],xy[m,1],s=110,color=cmap[l],alpha=0.45,linewidths=0)
    # 라벨: 군집 중앙값 위치, 5장 이상 L3만
    from adjustText import adjust_text
    texts=[]
    for l in sorted(set(labs)):
        m=np.array([x==l for x in labs])
        if m.sum()<5: continue
        cx,cy=np.median(xy[m,0]),np.median(xy[m,1])
        texts.append(ax.text(cx,cy,nd[l].get('label_en') or l,fontsize=9,fontweight='bold',
                    color=tuple(v*0.6 for v in cmap[l][:3]),ha='center',
                    bbox=dict(boxstyle='round,pad=0.15',fc='white',ec='none',alpha=0.7)))
    # 겹치는 라벨은 여백으로 밀어내고 중앙값 위치를 직선으로 연결
    adjust_text(texts,ax=ax,expand=(1.15,1.35),
                arrowprops=dict(arrowstyle='-',color='#777',lw=0.7,alpha=0.8))
    ax.set_axis_off()
    ax.set_title(f'{setname.upper()} — L4 risk-card landscape (UMAP of BGE-M3 bilingual embeddings, colored by L3)',fontsize=11)
    fig.tight_layout()
    fig.savefig(f'{OUT}/{setname}_landscape.png',bbox_inches='tight')
    plt.close(fig)
    print(setname,'->',f'{OUT}/{setname}_landscape.png', emb.shape)
    return xy,labs

if __name__=='__main__':
    for s in ('c80','c70'): build(s)
